In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Find the files

In [ ]:
import os
root = "/kaggle/input"
for dirpath, _, files in os.walk(root):
    for f in files:
        p = os.path.join(dirpath, f)
        print(p, f"{os.path.getsize(p)/1e6:.1f} MB")

# Columns, shapes, and missing values for every file

In [ ]:
import pandas as pd, glob, os

base = "/kaggle/input/datasets/prishabhkumar/processed-data/processed"
files = sorted(glob.glob(f"{base}/**/*.csv", recursive=True))

dfs = {}
for p in files:
    name = os.path.basename(p).replace(".csv", "")
    dfs[name] = pd.read_csv(p)
    d = dfs[name]
    print(f"\n=== {name} === shape={d.shape}")
    print("columns:", d.columns.tolist())
    print("nulls:", d.isna().sum()[d.isna().sum() > 0].to_dict())

# Look at raw rows from each source (repr shows newlines and hidden characters)

In [ ]:
for name in ["enron_train", "spamassassin_train", "intent_train"]:
    d = dfs[name]
    print("\n" + "=" * 90, f"\n{name}")
    print(d.head(3).to_string(max_colwidth=300))

# Text length stats

In [ ]:
TEXT_COL = "cleaned_body"
for name in ["enron_train", "spamassassin_train", "intent_train"]:
    w = dfs[name][TEXT_COL].astype(str).str.split().str.len()
    print(name, w.describe()[["mean", "50%", "max"]].round(0).to_dict(),
          "| over 400 words:", (w > 400).sum())

# How noisy the text still is

In [ ]:
import re
checks = {
    "headers (From:/To:/Subject:)": r"(?im)^\s*(from|to|cc|subject|sent|date)\s*:",
    "quoted lines (>)": r"(?m)^\s*>",
    "forwarded / original message": r"(?i)original message|forwarded by|-----",
    "directory names (/HOU/ECT@)": r"/[A-Z]{2,}/[A-Za-z]+@",
    "URLs": r"https?://",
    "email addresses": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "money": r"[$€£₹]\s?\d",
    "times (3:00 / 3 pm)": r"(?i)\b\d{1,2}:\d{2}\b|\b\d{1,2}\s?(am|pm)\b",
    "weekday/month names": r"(?i)\b(mon|tue|wed|thu|fri|sat|sun|jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\b",
}
for name in ["enron_train", "spamassassin_train", "intent_train"]:
    s = dfs[name][TEXT_COL].astype(str)
    print(f"\n{name}")
    for k, pat in checks.items():
        print(f"  {k}: {s.str.contains(pat, regex=True).mean()*100:.1f}%")

# Any existing entity labels, and the intent classes

In [ ]:
for name, d in dfs.items():
    cand = [c for c in d.columns if any(k in c.lower() for k in ["ent", "label", "tag", "ner", "span"])]
    if cand: print(name, "->", cand)

d = dfs["intent_train"]
for c in d.columns:
    if d[c].nunique() < 30:
        print(f"\n{c}:\n", d[c].value_counts())

# 01 · Silver-label Enron emails for NER

**Before running:** Settings → *Internet: On*, *Accelerator: GPU T4* (CPU works but is slow; see `LIMIT_EMAILS`).

**What this notebook does**
1. Loads the Enron train/val/test CSVs, cleans the text, and splits each email into short chunks (model-sized pieces).
2. Removes duplicate chunks, including across splits, so val/test don't leak from train.
3. Labels the **train** chunks automatically ("silver" labels): spaCy `en_core_web_trf` for PERSON / ORG / GPE / DATE / TIME / MONEY, plus rules for EMAIL / PHONE / URL, Enron directory names, and numeric dates/times. Rules win over the model.
4. Draws a small random sample of **val** and **test** chunks and exports them to Label Studio with the automatic labels pre-filled, so you only have to correct them. These become your gold dev/test sets.

**Outputs** (in `/kaggle/working/ner_data/`): `silver_train.jsonl`, `gold_val_ls.json`, `gold_test_ls.json`, `label_studio_config.xml`, and `ner_data.zip` with all of them.

In [3]:
import spacy

nlp = spacy.load("en_core_web_trf")

text = "Tanushka joined Tata Communications in Mumbai on 15 May 2025."

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

Tanushka -> ORG
Tata Communications -> ORG
Mumbai -> GPE
15 May 2025 -> DATE


In [2]:
# Run once. If imports fail afterwards, restart the session (Run -> Restart) and skip this cell.
!pip install -q -U spacy-transformers
!python -m spacy download en_core_web_trf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 3.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import collections, hashlib, json, os, random, re, shutil

import pandas as pd
import spacy
from tqdm.auto import tqdm

# ---------------- settings ----------------
BASE = "/kaggle/input/datasets/prishabhkumar/processed-data/processed"
PATHS = {s: f"{BASE}/{s}/enron_{s}.csv" for s in ("train", "val", "test")}
OUT = "/kaggle/working/ner_data"
os.makedirs(OUT, exist_ok=True)

MODEL_NAME = "en_core_web_trf"
CHUNK_CHARS = 900            # target chunk size (~200 tokens), keeps training sequences short
MAX_CHUNKS_PER_EMAIL = 4     # very long emails are mostly forwarded junk; keep the first few chunks
MIN_CHUNK_CHARS = 30
LIMIT_EMAILS = {"train": None, "val": None, "test": None}   # e.g. {"train": 2000, ...} if running on CPU
GOLD_CHUNKS = {"val": 150, "test": 150}                     # chunks to hand-correct per split
SEED = 42

LABELS = ["PERSON", "ORG", "GPE", "DATE", "TIME", "MONEY", "EMAIL", "PHONE", "URL"]
MODEL_LABELS = {"PERSON", "ORG", "GPE", "DATE", "TIME", "MONEY"}   # labels we accept from spaCy
random.seed(SEED)

## Cleaning and chunking
Quoted reply lines (starting with `>`) are dropped, whitespace is normalised, and each email is cut into chunks at paragraph / sentence boundaries. All labels refer to the **chunk text**, so the chunk text is what gets saved.

In [5]:
def normalize(text):
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    lines = [ln.rstrip() for ln in text.split("\n") if not ln.lstrip().startswith(">")]
    text = "\n".join(lines)
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")


def _pieces(par, limit):
    """Split one paragraph into pieces no longer than `limit` characters."""
    if len(par) <= limit:
        return [par]
    out, cur = [], ""
    for s in _SENT_SPLIT.split(par):
        while len(s) > limit:                       # long run with no sentence breaks
            cut = s.rfind(" ", 0, limit)
            cut = cut if cut > 0 else limit
            if cur:
                out.append(cur)
                cur = ""
            out.append(s[:cut])
            s = s[cut:].lstrip()
        if cur and len(cur) + len(s) + 1 > limit:
            out.append(cur)
            cur = s
        else:
            cur = f"{cur} {s}".strip()
    if cur:
        out.append(cur)
    return out


def chunk_email(text, limit=CHUNK_CHARS, max_chunks=MAX_CHUNKS_PER_EMAIL):
    chunks, cur = [], ""
    for par in re.split(r"\n\s*\n", normalize(text)):
        par = par.strip()
        if not par:
            continue
        for piece in _pieces(par, limit):
            if cur and len(cur) + len(piece) + 2 > limit:
                chunks.append(cur)
                cur = piece
            else:
                cur = f"{cur}\n\n{piece}" if cur else piece
    if cur:
        chunks.append(cur)
    return [c for c in chunks if len(c) >= MIN_CHUNK_CHARS][:max_chunks]


def text_hash(t):
    return hashlib.md5(re.sub(r"\W+", " ", t.lower()).strip().encode()).hexdigest()


def build_chunks(split):
    df = pd.read_csv(PATHS[split])
    if LIMIT_EMAILS[split]:
        df = df.sample(min(LIMIT_EMAILS[split], len(df)), random_state=SEED)
    rows = []
    for file, body in zip(df["file"], df["cleaned_body"]):
        for i, c in enumerate(chunk_email(body)):
            rows.append({"id": f"{file}#{i}", "file": file, "split": split, "text": c})
    return rows


chunks = {s: build_chunks(s) for s in PATHS}

# Drop duplicate chunks inside a split and across splits (train keeps priority).
seen = set()
for s in ("train", "val", "test"):
    kept = []
    for r in chunks[s]:
        k = text_hash(r["text"])
        if k not in seen:
            seen.add(k)
            kept.append(r)
    print(f"{s}: {len(chunks[s])} chunks -> {len(kept)} after de-duplication")
    chunks[s] = kept

train: 12104 chunks -> 11911 after de-duplication
val: 2533 chunks -> 2447 after de-duplication
test: 2623 chunks -> 2546 after de-duplication


## Rules
Regexes for EMAIL / PHONE / URL / MONEY / TIME / numeric DATE, plus Enron directory names such as `Andrew S Fastow/HOU/ECT@ECT`. These always win over the model when spans overlap.

In [6]:
URL_RE = re.compile(r"(?:https?://|www\.)[^\s<>\"]+", re.I)

RULES = [
    ("URL", URL_RE),
    ("EMAIL", re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")),
    ("PHONE", re.compile(r"(?<![\w.])(?:\+?1[\s.-])?\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}(?!\d)")),
    ("MONEY", re.compile(
        r"[$€£]\s?\d[\d,]*(?:\.\d+)?(?:\s?(?:million|billion|thousand|mm|bn|[kmb])\b)?", re.I)),
    ("TIME", re.compile(
        r"\b(?:[01]?\d|2[0-3]):[0-5]\d(?::[0-5]\d)?(?:\s?[ap]\.?m\.?)?(?:\s(?:[CEMP][SD]?T|GMT|UTC)\b)?"
        r"|\b(?:1[0-2]|[1-9])\s?[ap]\.?m\.?(?!\w)", re.I)),
    ("DATE", re.compile(
        r"\b(?:0?[1-9]|1[0-2])[/-](?:0?[1-9]|[12]\d|3[01])[/-](?:\d{4}|\d{2})\b|\b\d{4}-\d{2}-\d{2}\b")),
]
_TRAIL = ".,;:!?)]}'\""

# Enron directory names. Words are separated by spaces/tabs only (never newlines).
# The path-only form needs >= 2 capitalised segments (/HOU/ECT) so "Sales Team/Support" is not a person.
_NAME_WORD = r"[A-Z][A-Za-z.'-]*"
_NAME = rf"({_NAME_WORD}(?:[ \t]+{_NAME_WORD}){{1,4}})"
_SUFFIX_AT = r"(?:/[A-Za-z0-9_&.-]+)*(?:@[A-Za-z0-9_&.-]+)+"
_SUFFIX_PATH = r"(?:/[A-Z][A-Za-z0-9_&-]{1,9}){2,}"
_ENRON = [re.compile(_NAME + r"[ \t]*" + _SUFFIX_AT), re.compile(_NAME + r"[ \t]*" + _SUFFIX_PATH)]
_LEAD_NOISE = {"cc", "bcc", "to", "from", "by", "hi", "hello", "dear", "thanks", "thank",
               "regards", "forwarded", "attn"}
_LEAD_WORD = re.compile(r"([A-Za-z]+)[ \t]+")


def enron_directory_entities(text):
    found = []
    for rx in _ENRON:
        for m in rx.finditer(text):
            name, start = m.group(1), m.start(1)
            while True:                                    # drop "Cc", "Thanks", ... in front of the name
                lw = _LEAD_WORD.match(name)
                if lw and lw.group(1).lower() in _LEAD_NOISE:
                    name = name[lw.end():]
                    start += lw.end()
                else:
                    break
            name = name.rstrip()
            if len(name.split()) < 2:
                continue
            found.append({"start": start, "end": start + len(name), "label": "PERSON",
                          "full_start": m.start(), "full_end": m.end()})
    found.sort(key=lambda e: (e["full_start"], -(e["full_end"] - e["full_start"])))
    kept, last = [], -1
    for e in found:
        if e["full_start"] >= last:
            kept.append(e)
            last = e["full_end"]
    return kept


def rule_entities(text):
    out = []
    for label, rx in RULES:
        for m in rx.finditer(text):
            s, e = m.start(), m.end()
            if label == "URL":
                while e > s and text[e - 1] in _TRAIL:
                    e -= 1
            out.append({"start": s, "end": e, "label": label})
    return out + enron_directory_entities(text)

## Cleaning up model predictions
spaCy's output is filtered for the usual email noise (meeting IDs tagged as DATE, tools like Zoom tagged as PERSON, template words, URLs), and rule entities are given priority when spans overlap.

In [7]:
_ARTICLE = re.compile(r"^(?:a|an|the)\s+", re.I)
_URLISH = re.compile(r"https?://|www\.|\.com/|\.zoom\.us|\.org/|\.edu/", re.I)
_TOOLS = re.compile(r"\b(?:zoom|teams|slack|loopscheduler)\b", re.I)
_TOOLS_EXACT = {"webex", "skype", "outlook", "gmail", "google meet", "google calendar", "chime"}
_BOILER = {"candidate", "interviewer", "recruiter", "hiring manager", "off campus", "on campus",
           "virtual interview", "interview preparation", "mm am", "hh:mm", "passcode", "meeting id"}
_BOILER_RE = re.compile(r"campus|mm am")
_YEAR = re.compile(r"(?:19|20)\d{2}")
_NUMERIC_ONLY = re.compile(r"[\d\s\-]+")
_DATE_NUMERIC = re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b\d{4}-\d{2}-\d{2}\b")


def valid_date(t):
    """Reject DATE entities that are really meeting IDs, passcodes or phone numbers."""
    if t.isdigit():
        return bool(_YEAR.fullmatch(t))                       # a bare year is fine, long digit runs are not
    if _NUMERIC_ONLY.fullmatch(t):
        digits = re.sub(r"[\s\-]", "", t)
        if len(digits) > 5 and not _DATE_NUMERIC.search(t):
            return False
    return True


def sanitize(text, label, start):
    """Return a cleaned entity dict (offsets exact), or None to discard."""
    s = text.lstrip()
    start += len(text) - len(s)
    text = s
    m = _ARTICLE.match(text)
    if m:
        text = text[m.end():]
        start += m.end()
    text = text.rstrip()
    if not text:
        return None
    low = text.lower()
    if _URLISH.search(text):
        return None
    if label == "DATE" and not valid_date(text):
        return None
    if label == "PERSON" and (
        len(text) <= 1 or re.search(r"\d", text) or low in _TOOLS_EXACT or low in _BOILER or low in _LEAD_NOISE
        or _TOOLS.search(text) or _BOILER_RE.search(low)
    ):
        return None
    if label == "ORG" and low in _BOILER:
        return None
    return {"start": start, "end": start + len(text), "label": label}


def _block(e):
    return e.get("full_start", e["start"]), e.get("full_end", e["end"])


def resolve(rule_ents, model_ents, text):
    """Rule entities win; within a tier the longer span wins. Returns entities sorted by position."""
    cands = [(0, e) for e in rule_ents] + [(1, e) for e in model_ents]
    cands.sort(key=lambda c: (c[0], -(c[1]["end"] - c[1]["start"]), c[1]["start"]))
    kept = []
    for _, e in cands:
        lo, hi = _block(e)
        if not any(lo < _block(k)[1] and _block(k)[0] < hi for k in kept):
            kept.append(e)
    kept.sort(key=lambda e: e["start"])
    return [{"start": e["start"], "end": e["end"], "label": e["label"],
             "text": text[e["start"]:e["end"]]} for e in kept]


def load_nlp():
    try:
        spacy.prefer_gpu()
    except Exception:
        pass
    # NER only needs the transformer + ner components; the rest just slows things down.
    return spacy.load(MODEL_NAME, disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])


def label_rows(rows, nlp, batch_size=32):
    """Add an `entities` list (character offsets into row['text']) to every row."""
    masked = [URL_RE.sub(lambda m: " " * len(m.group()), r["text"]) for r in rows]   # same length -> offsets stay valid
    for r, doc in zip(rows, tqdm(nlp.pipe(masked, batch_size=batch_size), total=len(rows))):
        model = []
        for ent in doc.ents:
            if ent.label_ in MODEL_LABELS:
                clean = sanitize(ent.text, ent.label_, ent.start_char)
                if clean:
                    model.append(clean)
        r["entities"] = resolve(rule_entities(r["text"]), model, r["text"])
    return rows

## Run the labeling
Train chunks are labeled in full. For val and test, only a random sample (`GOLD_CHUNKS`) is labeled, because those are the chunks you will correct by hand.

In [8]:
nlp = load_nlp()
print("pipeline:", nlp.pipe_names)

silver_train = label_rows(chunks["train"], nlp)

gold = {}
for s in ("val", "test"):
    pool = chunks[s]
    sample = random.Random(SEED).sample(pool, min(GOLD_CHUNKS[s], len(pool)))
    gold[s] = label_rows(sample, nlp)

pipeline: ['transformer', 'ner']


  0%|          | 0/11911 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s]

In [10]:
VAGUE_DATES = {"now", "then", "monthly", "daily", "weekly", "quarterly", "yearly", "annual", "annually",
               "currently", "recently", "once", "twice"}
_POSSESSIVE = re.compile(r"['’]s$")

def postprocess(ents):
    out = []
    for e in ents:
        if e["label"] == "DATE" and e["text"].lower() in VAGUE_DATES:
            continue
        if e["label"] in ("PERSON", "ORG", "GPE") and _POSSESSIVE.search(e["text"]):
            e = {**e, "end": e["end"] - 2, "text": e["text"][:-2]}
        out.append(e)
    return out

for rows in (silver_train, gold["val"], gold["test"]):
    for r in rows:
        r["entities"] = postprocess(r["entities"])

## Inspect the result
Check the label counts and read a few examples. Entities appear as `[[text|LABEL]]`. This is the point to catch systematic mistakes (a label that is almost never found, or one that is wrong most of the time).

In [11]:
def show(r):
    t = r["text"]
    for e in sorted(r["entities"], key=lambda e: -e["start"]):
        t = t[:e["start"]] + f"[[{e['text']}|{e['label']}]]" + t[e["end"]:]
    return t


for name, rows in [("silver_train", silver_train), ("gold_val", gold["val"]), ("gold_test", gold["test"])]:
    cnt = collections.Counter(e["label"] for r in rows for e in r["entities"])
    with_ent = sum(1 for r in rows if r["entities"]) / max(len(rows), 1)
    print(f"{name}: {len(rows)} chunks, {with_ent:.0%} with >=1 entity")
    print("   ", {l: cnt.get(l, 0) for l in LABELS})

for r in random.Random(1).sample(silver_train, 6):
    print("\n" + "-" * 80 + f"\n{r['id']}\n" + show(r))

silver_train: 11911 chunks, 96% with >=1 entity
    {'PERSON': 56946, 'ORG': 27123, 'GPE': 7539, 'DATE': 22173, 'TIME': 10803, 'MONEY': 3428, 'EMAIL': 5218, 'PHONE': 2508, 'URL': 1703}
gold_val: 150 chunks, 95% with >=1 entity
    {'PERSON': 669, 'ORG': 269, 'GPE': 99, 'DATE': 259, 'TIME': 144, 'MONEY': 22, 'EMAIL': 86, 'PHONE': 25, 'URL': 21}
gold_test: 150 chunks, 97% with >=1 entity
    {'PERSON': 804, 'ORG': 447, 'GPE': 76, 'DATE': 266, 'TIME': 136, 'MONEY': 40, 'EMAIL': 64, 'PHONE': 34, 'URL': 37}

--------------------------------------------------------------------------------
kaminski-v/deleted_items/31.#2
Search criteria include job category, industry expertise/skills requirements, experience desired, education desired, languages required, and job location. Full job access: Complete job detail is available for all jobs. Personalized career center: When you place your resume in [[Rigzone|ORG]]'s [[Career Center|ORG]] you are automatically provided with a personalized page that a

## Save everything
`gold_*_ls.json` are Label Studio tasks with the automatic labels as pre-annotations. Download `ner_data.zip` from the notebook's **Output** tab (use *Save Version* → *Save & Run All*, or the file browser on the right).

In [12]:
def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def to_label_studio(rows):
    tasks = []
    for r in rows:
        result = [{"from_name": "label", "to_name": "text", "type": "labels",
                   "value": {"start": e["start"], "end": e["end"], "text": e["text"], "labels": [e["label"]]}}
                  for e in r["entities"]]
        tasks.append({"data": {"text": r["text"], "id": r["id"]},
                      "predictions": [{"model_version": "silver-v1", "result": result}]})
    return tasks


save_jsonl(silver_train, f"{OUT}/silver_train.jsonl")
for s in ("val", "test"):
    with open(f"{OUT}/gold_{s}_ls.json", "w", encoding="utf-8") as f:
        json.dump(to_label_studio(gold[s]), f, ensure_ascii=False, indent=1)

config = "<View>\n  <Labels name=\"label\" toName=\"text\">\n" + "".join(
    f"    <Label value=\"{l}\"/>\n" for l in LABELS) + "  </Labels>\n  <Text name=\"text\" value=\"$text\"/>\n</View>\n"
with open(f"{OUT}/label_studio_config.xml", "w") as f:
    f.write(config)

shutil.make_archive("/kaggle/working/ner_data", "zip", OUT)
print(os.listdir(OUT))
print(config)

['silver_train.jsonl', 'gold_test_ls.json', 'gold_val_ls.json', 'label_studio_config.xml']
<View>
  <Labels name="label" toName="text">
    <Label value="PERSON"/>
    <Label value="ORG"/>
    <Label value="GPE"/>
    <Label value="DATE"/>
    <Label value="TIME"/>
    <Label value="MONEY"/>
    <Label value="EMAIL"/>
    <Label value="PHONE"/>
    <Label value="URL"/>
  </Labels>
  <Text name="text" value="$text"/>
</View>



# Testing

In [13]:
# --- patch: a sentence-ending "." is no longer swallowed by TIME ("03:59 PM." -> "03:59 PM") ---
_rule_entities_v1 = rule_entities
def rule_entities(text):
    out = _rule_entities_v1(text)
    for e in out:
        if e["label"] == "TIME" and text[e["end"] - 1] == "." and text[e["start"]:e["end"]].count(".") == 1:
            e["end"] -= 1
    return out


# --- test function ---
def test_ner(text, nlp=nlp):
    """Run the labeling pipeline (rules + spaCy) on any text. Prints it marked up and returns [(text, label), ...]."""
    pieces = chunk_email(text) if len(text) > CHUNK_CHARS else [normalize(text)]
    rows = label_rows([{"id": f"demo#{i}", "text": p} for i, p in enumerate(pieces)], nlp)
    found = []
    for r in rows:
        r["entities"] = postprocess(r["entities"])      # needs the postprocess cell from my last message
        marked = r["text"]
        for e in sorted(r["entities"], key=lambda e: -e["start"]):
            marked = marked[:e["start"]] + f"[[{e['text']}|{e['label']}]]" + marked[e["end"]:]
        print(marked)
        found += [(e["text"], e["label"]) for e in r["entities"]]
    print()
    for t, l in found:
        print(f"{t!r:38} {l}")
    return found


test_ner("Hi Vince, can we meet on 05/04/2001 at 3:30 pm CST? Budget is $2.5 million. "
         "Call Sara at (713) 853-5620 or sara@enron.com. See http://www.enron.com/x. "
         "Forwarded by Steven J Kean/NA/Enron on 07/24/2000 03:59 PM. Reports are monthly now.")

  0%|          | 0/1 [00:00<?, ?it/s]

Hi [[Vince|PERSON]], can we meet on [[05/04/2001|DATE]] at [[3:30 pm CST|TIME]]? Budget is [[$2.5 million|MONEY]]. Call [[Sara|PERSON]] at [[(713) 853-5620|PHONE]] or [[sara@enron.com|EMAIL]]. See [[http://www.enron.com/x|URL]]. Forwarded by [[Steven J Kean|PERSON]]/NA/Enron on [[07/24/2000|DATE]] [[03:59 PM|TIME]]. Reports are monthly now.

'Vince'                                PERSON
'05/04/2001'                           DATE
'3:30 pm CST'                          TIME
'$2.5 million'                         MONEY
'Sara'                                 PERSON
'(713) 853-5620'                       PHONE
'sara@enron.com'                       EMAIL
'http://www.enron.com/x'               URL
'Steven J Kean'                        PERSON
'07/24/2000'                           DATE
'03:59 PM'                             TIME


[('Vince', 'PERSON'),
 ('05/04/2001', 'DATE'),
 ('3:30 pm CST', 'TIME'),
 ('$2.5 million', 'MONEY'),
 ('Sara', 'PERSON'),
 ('(713) 853-5620', 'PHONE'),
 ('sara@enron.com', 'EMAIL'),
 ('http://www.enron.com/x', 'URL'),
 ('Steven J Kean', 'PERSON'),
 ('07/24/2000', 'DATE'),
 ('03:59 PM', 'TIME')]

# 02 · Train a small NER model → quantize to int8 → check size and accuracy

Run notebook 01 first (it creates `silver_train.jsonl`). **Settings:** Internet *On*, Accelerator *GPU T4*.

**What this notebook does**
1. Trains a small transformer (`distilbert-base-cased` by default) for token classification on the silver labels.
2. Scores it. On your hand-corrected gold file if you have one, otherwise on a held-out slice of the silver data (that only shows how well it copies the automatic labels).
3. Exports it to ONNX and quantizes the weights to int8.
4. Prints a **size table** (fp32 vs int8, plus tokenizer files) and compares **accuracy and speed** before and after quantization, so you can see what the compression cost.
5. Saves an `android/` folder (int8 model, tokenizer files, label list) and defines `test_ner(text)` that runs the quantized model.

In [14]:
!pip install -q onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.7 MB/s eta 0:00:00:00:0100:01


In [15]:
import collections, inspect, json, math, os, random, re, shutil, statistics, time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForTokenClassification, AutoTokenizer, get_linear_schedule_with_warmup

# ---------------- settings ----------------
SILVER_PATH = "/kaggle/working/ner_data/silver_train.jsonl"            # from notebook 01
GOLD_PATH = "/kaggle/working/ner_data/gold_val_corrected.json"         # Label Studio JSON export (optional, see below)

# distilbert-base-cased : ~65 MB after int8, keeps capitalisation (helps PERSON / ORG)
# google/mobilebert-uncased : ~26 MB after int8, but lower-cased (usually a few F1 points worse on names)
BASE_MODEL = "distilbert-base-cased"
MAX_LEN = 320
EPOCHS, BATCH, LR, WARMUP = 3, 32, 5e-5, 0.06
OUT_DIR = "/kaggle/working/ner_model"
SEED = 42

LABELS = ["PERSON", "ORG", "GPE", "DATE", "TIME", "MONEY", "EMAIL", "PHONE", "URL"]
LABEL_LIST = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
ID2LABEL = dict(enumerate(LABEL_LIST))
LABEL2ID = {l: i for i, l in ID2LABEL.items()}

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## Data and tokenization
Silver chunks are split **by email** into train and a small dev slice. Character-offset entities become BIO tags on word pieces (every piece of an entity gets `B-`/`I-`; special tokens are ignored).

In [16]:
rows = [json.loads(l) for l in open(SILVER_PATH, encoding="utf-8")]
files = sorted({r["file"] for r in rows})
random.Random(SEED).shuffle(files)
dev_files = set(files[: max(1, int(0.05 * len(files)))])
train_rows = [r for r in rows if r["file"] not in dev_files]
dev_rows = [r for r in rows if r["file"] in dev_files]
print(f"train chunks: {len(train_rows)} | dev chunks: {len(dev_rows)}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABEL_LIST), id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True)

INPUT_NAMES = [n for n in ("input_ids", "attention_mask", "token_type_ids")
               if n in inspect.signature(model.forward).parameters]
NEEDS_TTI = "token_type_ids" in INPUT_NAMES
print("model inputs:", INPUT_NAMES)


def encode(rows):
    enc = tokenizer([r["text"] for r in rows], truncation=True, max_length=MAX_LEN,
                    return_offsets_mapping=True, return_token_type_ids=NEEDS_TTI)
    feats = []
    for i, r in enumerate(rows):
        ents = sorted(r["entities"], key=lambda e: e["start"])
        labels, j, prev = [], 0, None
        for s, e in enc["offset_mapping"][i]:
            if s == e:                                    # [CLS], [SEP], padding
                labels.append(-100)
                continue
            while j < len(ents) and ents[j]["end"] <= s:
                j += 1
            if j < len(ents) and ents[j]["start"] <= s < ents[j]["end"]:
                labels.append(LABEL2ID[f"{'B' if prev != j else 'I'}-{ents[j]['label']}"])
                prev = j
            else:
                labels.append(LABEL2ID["O"])
                prev = None
        f = {k: enc[k][i] for k in INPUT_NAMES}
        f["labels"] = labels
        feats.append(f)
    return feats


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    out = {k: torch.tensor([b[k] + [0] * (n - len(b[k])) for b in batch]) for k in INPUT_NAMES if k != "input_ids"}
    out["input_ids"] = torch.tensor([b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch])
    out["labels"] = torch.tensor([b["labels"] + [-100] * (n - len(b["labels"])) for b in batch])
    return out


train_feats = encode(train_rows)
loader = DataLoader(train_feats, batch_size=BATCH, shuffle=True, collate_fn=collate)
print("example label ids:", train_feats[0]["labels"][:25])

train chunks: 11309 | dev chunks: 602


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model inputs: ['input_ids', 'attention_mask']
example label ids: [-100, 1, 2, 0, 1, 2, 0, 1, 2, 2, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 4, 0, 1, 2]


## Train
Plain PyTorch loop (mixed precision on GPU). On a T4 this takes a few minutes per epoch.

In [17]:
model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(loader)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

for epoch in range(EPOCHS):
    model.train()
    running, t0 = 0.0, time.time()
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            loss = model(**batch).loss
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        running += loss.item()
        if step % 100 == 0:
            print(f"epoch {epoch+1} step {step}/{len(loader)} loss {running/100:.4f}")
            running = 0.0
    print(f"epoch {epoch+1} done in {time.time()-t0:.0f}s")

model.eval().cpu()
FP32_DIR = f"{OUT_DIR}/fp32"
model.save_pretrained(FP32_DIR)
tokenizer.save_pretrained(FP32_DIR)
json.dump(LABEL_LIST, open(f"{FP32_DIR}/labels.json", "w"))
print("saved", FP32_DIR)

/tmp/ipykernel_178/1872237628.py:21: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


epoch 1 step 100/354 loss 0.9186
epoch 1 step 200/354 loss 0.1620
epoch 1 step 300/354 loss 0.1324
epoch 1 done in 86s
epoch 2 step 100/354 loss 0.1016
epoch 2 step 200/354 loss 0.0953
epoch 2 step 300/354 loss 0.0963
epoch 2 done in 85s
epoch 3 step 100/354 loss 0.0725
epoch 3 step 200/354 loss 0.0761
epoch 3 step 300/354 loss 0.0741
epoch 3 done in 85s
saved /kaggle/working/ner_model/fp32


## Prediction and scoring (shared by the PyTorch and ONNX models)
Spans are decoded straight from token offsets, so results are character spans in the original text. Scoring is **exact match** at entity level: both boundaries and the label must be right.

**Gold file:** after you correct the val chunks in Label Studio, export as *JSON* (not JSON-MIN) and put it at `GOLD_PATH`. If the file is missing, the notebook falls back to the silver dev slice.

In [18]:
def decode(text, offsets, pred_ids):
    spans, cur = [], None
    for (s, e), p in zip(offsets, pred_ids):
        if s == e:
            continue
        tag = ID2LABEL[int(p)]
        if tag == "O":
            if cur:
                spans.append(cur)
                cur = None
            continue
        bio, lab = tag.split("-", 1)
        if cur and bio == "I" and cur["label"] == lab:
            cur["end"] = e
        else:
            if cur:
                spans.append(cur)
            cur = {"start": s, "end": e, "label": lab}
    if cur:
        spans.append(cur)
    for sp in spans:
        sp["text"] = text[sp["start"]:sp["end"]]
    return spans


def predict_spans(texts, run_logits, bs=32):
    """run_logits(dict of numpy arrays) -> logits array [batch, seq, labels]."""
    out = []
    for i in range(0, len(texts), bs):
        batch = texts[i:i + bs]
        enc = tokenizer(batch, truncation=True, max_length=MAX_LEN, padding=True,
                        return_offsets_mapping=True, return_tensors="np", return_token_type_ids=NEEDS_TTI)
        preds = run_logits({k: enc[k] for k in INPUT_NAMES}).argmax(-1)
        for b, text in enumerate(batch):
            keep = enc["attention_mask"][b].astype(bool)
            out.append(decode(text, enc["offset_mapping"][b][keep], preds[b][keep]))
    return out


def torch_runner(m):
    @torch.no_grad()
    def run(feed):
        return m(**{k: torch.from_numpy(v) for k, v in feed.items()}).logits.numpy()
    return run


def score(ref_rows, pred_spans, verbose=True):
    tp, fp, fn = (collections.Counter() for _ in range(3))
    for r, preds in zip(ref_rows, pred_spans):
        g = {(e["start"], e["end"], e["label"]) for e in r["entities"]}
        p = {(e["start"], e["end"], e["label"]) for e in preds}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    def prf(t, f_p, f_n):
        pr = t / (t + f_p) if t + f_p else 0.0
        rc = t / (t + f_n) if t + f_n else 0.0
        return pr, rc, (2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    table = []
    for l in LABELS:
        pr, rc, f1 = prf(tp[l], fp[l], fn[l])
        table.append({"label": l, "support": tp[l] + fn[l], "precision": pr, "recall": rc, "f1": f1})
    pr, rc, f1 = prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    table.append({"label": "MICRO", "support": sum(tp.values()) + sum(fn.values()), "precision": pr, "recall": rc, "f1": f1})
    df = pd.DataFrame(table).set_index("label").round(3)
    if verbose:
        print(df.to_string())
    return f1


def load_label_studio(path):
    tasks = json.load(open(path, encoding="utf-8"))
    out, skipped = [], 0
    for t in tasks:
        anns = [a for a in t.get("annotations", []) if not a.get("was_cancelled")]
        if not anns:
            skipped += 1
            continue
        ents = [{"start": r["value"]["start"], "end": r["value"]["end"], "label": r["value"]["labels"][0],
                 "text": r["value"].get("text", "")}
                for r in anns[0]["result"] if r.get("type") == "labels"]
        out.append({"id": t["data"].get("id"), "text": t["data"]["text"], "entities": ents})
    if skipped:
        print(f"note: {skipped} tasks have no annotation and were skipped")
    return out


if os.path.exists(GOLD_PATH):
    EVAL_ROWS, EVAL_NAME = load_label_studio(GOLD_PATH), "gold (hand-corrected)"
else:
    EVAL_ROWS, EVAL_NAME = dev_rows, "silver dev slice (agreement with automatic labels only)"
EVAL_TEXTS = [r["text"] for r in EVAL_ROWS]
print(f"evaluating on: {EVAL_NAME}, {len(EVAL_ROWS)} chunks")

fp32_pred = predict_spans(EVAL_TEXTS, torch_runner(model))
print("\n=== PyTorch fp32 ===")
f1_torch = score(EVAL_ROWS, fp32_pred)

evaluating on: silver dev slice (agreement with automatic labels only), 602 chunks

=== PyTorch fp32 ===
        support  precision  recall     f1
label                                    
PERSON     3160      0.882   0.856  0.869
ORG        1419      0.710   0.709  0.709
GPE         461      0.754   0.790  0.771
DATE       1177      0.869   0.862  0.866
TIME        593      0.938   0.897  0.917
MONEY       256      0.854   0.824  0.839
EMAIL       145      0.932   0.952  0.942
PHONE       122      0.814   0.934  0.870
URL          90      0.854   0.911  0.882
MICRO      7423      0.841   0.831  0.836


## Export to ONNX and quantize to int8
Dynamic int8 quantization stores the weights as 8-bit integers (about 4x smaller) and runs integer matrix math on the CPU. It needs no calibration data.

In [19]:
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

ONNX_DIR = f"{OUT_DIR}/onnx"
os.makedirs(ONNX_DIR, exist_ok=True)
FP32_ONNX, INT8_ONNX = f"{ONNX_DIR}/model_fp32.onnx", f"{ONNX_DIR}/model_int8.onnx"


class _Wrap(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, *args):
        return self.m(*args, return_dict=False)[0]


sample = tokenizer(["Hello Vince, the meeting is on Friday at 3 pm."], return_tensors="pt",
                   return_token_type_ids=NEEDS_TTI)
args = tuple(sample[k] for k in INPUT_NAMES)
export_kwargs = dict(
    input_names=INPUT_NAMES, output_names=["logits"],
    dynamic_axes={**{k: {0: "batch", 1: "seq"} for k in INPUT_NAMES}, "logits": {0: "batch", 1: "seq"}},
    opset_version=17,
)
try:
    torch.onnx.export(_Wrap(model).eval(), args, FP32_ONNX, dynamo=False, **export_kwargs)   # torch >= 2.5
except TypeError:
    torch.onnx.export(_Wrap(model).eval(), args, FP32_ONNX, **export_kwargs)

quantize_dynamic(FP32_ONNX, INT8_ONNX, weight_type=QuantType.QInt8)
print("exported:", sorted(os.listdir(ONNX_DIR)))


def onnx_runner(path, threads=None):
    so = ort.SessionOptions()
    if threads:
        so.intra_op_num_threads = threads
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    def run(feed):
        return sess.run(["logits"], {k: v.astype(np.int64) for k, v in feed.items()})[0]
    run.session = sess
    return run

/tmp/ipykernel_178/2358569347.py:27: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(_Wrap(model).eval(), args, FP32_ONNX, dynamo=False, **export_kwargs)   # torch >= 2.5
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


exported: ['model_fp32.onnx', 'model_int8.onnx']


## Size report
Sizes include external weight files. "Android payload" is the int8 model plus the tokenizer vocabulary the app must ship with it.

In [21]:
def file_mb(path):
    """Size of a file plus any external-data companions (model.onnx.data etc.)."""
    d, base = os.path.dirname(path), os.path.basename(path)
    return sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d) if f.startswith(base)) / 1e6


def dir_mb(path, names):
    return sum(os.path.getsize(os.path.join(path, n)) for n in names if os.path.exists(os.path.join(path, n))) / 1e6


weights_fp32 = next((f"{FP32_DIR}/{n}" for n in ("model.safetensors", "pytorch_model.bin")
                     if os.path.exists(f"{FP32_DIR}/{n}")), None)
tok_files = [n for n in ("vocab.txt", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json")
             if os.path.exists(f"{FP32_DIR}/{n}")]
n_params = sum(p.numel() for p in model.parameters())

size_rows = [
    ("PyTorch fp32 weights", file_mb(weights_fp32) if weights_fp32 else float("nan")),
    ("ONNX fp32", file_mb(FP32_ONNX)),
    ("ONNX int8  <- ship this", file_mb(INT8_ONNX)),
    ("tokenizer files (vocab etc.)", dir_mb(FP32_DIR, tok_files)),
]
size_df = pd.DataFrame(size_rows, columns=["artifact", "size_MB"]).set_index("artifact").round(1)
print(f"parameters: {n_params/1e6:.1f}M")
print(size_df.to_string())
int8_mb, fp32_mb = file_mb(INT8_ONNX), file_mb(FP32_ONNX)
print(f"\nint8 is {fp32_mb/int8_mb:.1f}x smaller than ONNX fp32 ({fp32_mb:.1f} MB -> {int8_mb:.1f} MB)")
print(f"Android payload (int8 model + vocab): {int8_mb + dir_mb(FP32_DIR, ['vocab.txt']):.1f} MB")

parameters: 65.2M
                              size_MB
artifact                             
PyTorch fp32 weights            260.8
ONNX fp32                       260.9
ONNX int8  <- ship this          65.6
tokenizer files (vocab etc.)      0.9

int8 is 4.0x smaller than ONNX fp32 (260.9 MB -> 65.6 MB)
Android payload (int8 model + vocab): 65.8 MB


## Did quantization hurt? Accuracy and speed
Same evaluation set, three models. A small F1 drop (about 1 point or less) is normal. If int8 drops more, tell me the numbers and we can try per-channel quantization, keeping the sensitive layers in fp32, or quantization-aware training.

In [22]:
runners = {
    "PyTorch fp32": torch_runner(model),
    "ONNX fp32": onnx_runner(FP32_ONNX),
    "ONNX int8": onnx_runner(INT8_ONNX),
}
preds = {"PyTorch fp32": fp32_pred}
for name in ("ONNX fp32", "ONNX int8"):
    preds[name] = predict_spans(EVAL_TEXTS, runners[name])


def same_entities(a, b):
    tot = same = 0
    for pa, pb in zip(a, b):
        sa = {(e["start"], e["end"], e["label"]) for e in pa}
        sb = {(e["start"], e["end"], e["label"]) for e in pb}
        tot += len(sa | sb)
        same += len(sa & sb)
    return same / tot if tot else 1.0


# latency on a typical (median-length) chunk, single thread ~ a modest phone core
typical = sorted(EVAL_TEXTS, key=len)[len(EVAL_TEXTS) // 2]
enc = tokenizer([typical], truncation=True, max_length=MAX_LEN, return_tensors="np", return_token_type_ids=NEEDS_TTI)
feed = {k: enc[k].astype(np.int64) for k in INPUT_NAMES}


def median_ms(path, threads=1, n=30):
    sess = onnx_runner(path, threads).session
    for _ in range(3):
        sess.run(["logits"], feed)
    ts = []
    for _ in range(n):
        t = time.perf_counter()
        sess.run(["logits"], feed)
        ts.append((time.perf_counter() - t) * 1e3)
    return statistics.median(ts)


report = []
for name in runners:
    report.append({
        "model": name,
        "micro_F1": round(score(EVAL_ROWS, preds[name], verbose=False), 4),
        "same_entities_as_fp32": round(same_entities(preds["PyTorch fp32"], preds[name]), 4),
        "ms_per_chunk_1thread": (round(median_ms({"ONNX fp32": FP32_ONNX, "ONNX int8": INT8_ONNX}[name]), 1)
                                 if name != "PyTorch fp32" else None),
    })
print(f"evaluated on: {EVAL_NAME}; latency measured on a {enc['input_ids'].shape[1]}-token chunk, this machine's CPU")
print(pd.DataFrame(report).set_index("model").to_string())
print("\n=== ONNX int8, per label ===")
score(EVAL_ROWS, preds["ONNX int8"])

evaluated on: silver dev slice (agreement with automatic labels only); latency measured on a 216-token chunk, this machine's CPU
              micro_F1  same_entities_as_fp32  ms_per_chunk_1thread
model                                                              
PyTorch fp32    0.8359                 1.0000                   NaN
ONNX fp32       0.8359                 1.0000                 196.9
ONNX int8       0.8248                 0.8563                 144.0

=== ONNX int8, per label ===
        support  precision  recall     f1
label                                    
PERSON     3160      0.867   0.853  0.860
ORG        1419      0.696   0.695  0.696
GPE         461      0.725   0.800  0.761
DATE       1177      0.843   0.852  0.847
TIME        593      0.933   0.889  0.910
MONEY       256      0.859   0.832  0.845
EMAIL       145      0.926   0.952  0.939
PHONE       122      0.768   0.893  0.826
URL          90      0.863   0.911  0.886
MICRO      7423      0.825   0.825  0.8

0.8248450552411749

## Package for Android and test the quantized model
`android/` holds everything the app needs: the int8 model, the tokenizer files, and the label list. The Android side has to tokenize exactly like this tokenizer does (WordPiece with this vocab) and decode the BIO tags with the same logic as `decode()` above.

In [23]:
ANDROID_DIR = f"{OUT_DIR}/android"
os.makedirs(ANDROID_DIR, exist_ok=True)
shutil.copy(INT8_ONNX, f"{ANDROID_DIR}/model_int8.onnx")
for n in tok_files + ["labels.json"]:
    shutil.copy(f"{FP32_DIR}/{n}", f"{ANDROID_DIR}/{n}")
json.dump({"max_len": MAX_LEN, "inputs": INPUT_NAMES, "base_model": BASE_MODEL, "labels": LABEL_LIST},
          open(f"{ANDROID_DIR}/meta.json", "w"), indent=1)
shutil.make_archive("/kaggle/working/ner_android", "zip", ANDROID_DIR)
print({n: round(os.path.getsize(f"{ANDROID_DIR}/{n}") / 1e6, 2) for n in sorted(os.listdir(ANDROID_DIR))})


# ---- test function: runs the quantized model on any text ----
_int8_run = onnx_runner(INT8_ONNX)


def _normalize(text):
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    text = "\n".join(ln.rstrip() for ln in text.split("\n") if not ln.lstrip().startswith(">"))
    return re.sub(r"\n{3,}", "\n\n", re.sub(r"[ \t]{2,}", " ", text)).strip()


def _split_text(text, limit=900):
    chunks, cur = [], ""
    for p in (p.strip() for p in re.split(r"\n\s*\n", _normalize(text))):
        if not p:
            continue
        while len(p) > limit:
            cut = p.rfind(" ", 0, limit)
            cut = cut if cut > 0 else limit
            if cur:
                chunks.append(cur)
                cur = ""
            chunks.append(p[:cut])
            p = p[cut:].lstrip()
        if cur and len(cur) + len(p) + 2 > limit:
            chunks.append(cur)
            cur = p
        else:
            cur = f"{cur}\n\n{p}" if cur else p
    return chunks + ([cur] if cur else [])


def test_ner(text):
    """Run the int8 model on any text. Prints it marked up and returns [(text, label), ...]."""
    chunks = _split_text(text)
    found = []
    for chunk, spans in zip(chunks, predict_spans(chunks, _int8_run)):
        marked = chunk
        for e in sorted(spans, key=lambda e: -e["start"]):
            marked = marked[:e["start"]] + f"[[{e['text']}|{e['label']}]]" + marked[e["end"]:]
        print(marked)
        found += [(e["text"], e["label"]) for e in spans]
    print()
    for t, l in found:
        print(f"{t!r:38} {l}")
    return found


_ = test_ner("Hi Vince, can we meet on 05/04/2001 at 3:30 pm CST? Budget is $2.5 million. "
             "Call Sara at (713) 853-5620 or sara@enron.com. See http://www.enron.com/x. "
             "Forwarded by Steven J Kean/NA/Enron on 07/24/2000 03:59 PM.")

{'labels.json': 0.0, 'meta.json': 0.0, 'model_int8.onnx': 65.6, 'special_tokens_map.json': 0.0, 'tokenizer.json': 0.67, 'tokenizer_config.json': 0.0, 'vocab.txt': 0.21}
Hi [[Vince|PERSON]], can we meet on [[05/04/2001|DATE]] at [[3:30 pm CST|TIME]]? Budget is [[$2.5 million|MONEY]]. Call [[Sara|PERSON]] at [[(713) 853-5620|PHONE]] or [[sara@enron.com|EMAIL]]. See [[http://www.enron.com/x|URL]]. Forwarded by [[Steven J Kean|PERSON]]/NA/Enron on [[07/24/2000|DATE]] [[03:59 PM.|TIME]]

'Vince'                                PERSON
'05/04/2001'                           DATE
'3:30 pm CST'                          TIME
'$2.5 million'                         MONEY
'Sara'                                 PERSON
'(713) 853-5620'                       PHONE
'sara@enron.com'                       EMAIL
'http://www.enron.com/x'               URL
'Steven J Kean'                        PERSON
'07/24/2000'                           DATE
'03:59 PM.'                            TIME


In [24]:
import random
extra = [
    "Hi Vince,\n\nCan we meet on 05/04/2001 at 3:30 pm CST? Budget is $2.5 million.",
    "Call Sara at (713) 853-5620 or sara@enron.com. See http://www.enron.com/x.",
    "Forwarded by Steven J Kean/NA/Enron on 07/24/2000 03:59 PM.",
    "Résumé from José Muñoz, Zürich office -- Q3 report due tomorrow.",
    "no entities in this one, just plain words and a few numbers like 42.",
]
texts = extra + random.Random(0).sample(EVAL_TEXTS, min(20, len(EVAL_TEXTS)))

golden = {"max_len": MAX_LEN, "inputs": INPUT_NAMES, "labels": LABEL_LIST, "token_cases": [], "chunk_cases": []}
for text in texts:
    enc = tokenizer([text], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    spans = predict_spans([text], _int8_run)[0]
    golden["token_cases"].append({
        "text": text,
        "input_ids": [int(i) for i in enc["input_ids"][0]],
        "offsets": [[int(a), int(b)] for a, b in enc["offset_mapping"][0]],
        "entities": [{"start": int(s["start"]), "end": int(s["end"]), "label": s["label"], "text": s["text"]}
                     for s in spans],
    })

# chunking cases: long raw emails (with quoted '>' lines and blank-line runs) and the chunks they must produce
raws = [
    "\n\n".join(texts[5:9]) + "\n> quoted reply line\n\n\n\nThanks,   Bob",
    " ".join(texts[5:12]),
    "word " * 400,
]
for raw in raws:
    golden["chunk_cases"].append({"raw": raw, "chunks": _split_text(raw)})

json.dump(golden, open(f"{ANDROID_DIR}/golden_tests.json", "w"), ensure_ascii=False, indent=1)
shutil.make_archive("/kaggle/working/ner_android", "zip", ANDROID_DIR)   # re-zip so the golden file is included
print(len(golden["token_cases"]), "token cases,", len(golden["chunk_cases"]), "chunk cases saved")

25 token cases, 3 chunk cases saved
